## 1. Project Setup

### i. Working with Existing Project on GitHub

1. Clone the repository to your local machine:
```bash
git clone <repository_url>
```

2. To create a new branch to work on different features or bug fixes without affecting the main project:
```bash
git branch <new_branch_name> # Create a branch
git checkout <new_branch_name> # Switch to the new branch
git checkout tags/<tag_name> # Switch using tags eg: git checkout tags/v1.0.0
```

3. To add, commit and push your changes to the remote repository:
```bash
git add .
git commit -m "Your commit message here"
git push origin <new_branch_name> # Push the branch to remote repository
```

4. Merging the branch with a Pull Request (PR):
- Go to the repository on GitHub.
- Click the **Pull requests** tab.
- Click the **New pull request** button.
- Select the branches you want to compare.
- Write a description and click **Create pull request**.

5. Deleting the branch after merging:
```bash
git branch -d TheSoftMax_test # To delete the local branch after the successful merging of the local branch to the main branch
git branch -D TheSoftMax_test # To delete without merge to the main branch
git push origin --delete TheSoftMax_test # To delete the remote branch on GitHub when the local is already deleted
git branch -a # To check your branches on local and remote
git push origin --delete TheSoftMax_test # Delete the remote branch
git fetch -p # To fetch the current status of all branches
```

### ii. Creating New Project [DVC](https://dvc.org/doc/install), Git & [Cookiecutter](https://cookiecutter-data-science.drivendata.org/)

1. Install Cookiecutter and DVC using pip:
```bash
pip install cookiecutter-data-science # Installing Cookiecutter using pip
pip install dvc[gs] # installing dvc for Google Storage, for AWS use dvc[s3]
```

2. Create a new project structure using Cookiecutter template:
```bash
ccds # Run the Cookiecutter template, fill in your project details
cd your_project_name # move into the newly created directory
```

3. Create new virtual environment and activate it:
```bash
conda create -n myenv python=3.15.5 # creating new virtual environment with Python 3.15.5
conda activate myenv # activate the virtual environment
pip install -r requirements.txt # install dependencies from `requirements.txt`
```

4. Initialize Git and DVC in the project directory:
```bash
git init # Initialize Git
dvc init # Initialize DVC and creates a **.dvcignore** file and **.dvc** internal directory that contains `cache`.
dvc config core.autostage true # Enable auto-staging of changes by DVC
dvc install # To enable auto pull of data on git checkout. This sets up Git hooks so that every time you git checkout a commit, DVC automatically runs dvc pull behind the scenes.
```

5. Add data and models folders to DVC tracking and commit everything to Git:
```bash
dvc add data/raw # add data folder to DVC tracking
dvc status # To verify the status of DVC tracking
git add . # Stage the CCDS scaffolding, the DVC config, and your new data/raw.dvc file to Git all at once.
git commit -m "ccds template with git and dvc initialized" # Commit Everything
# Push this repo to GitHub using VS Code Git extension.
```

DVC will add data and models folders to your .gitignore file and track these folder by itself. It also creates data/raw.dvc and models.dvc files which are tracked by Git. Content of data folder will be added to the DVC cache.

> NOTE: `DVC add file_name` don't require if we are using dvc pipeline as dvc automatically tracks the output files, metrics, and parameters. This changes are tracked by dvc.lock file instead of file_name.dvc file.

## 2. DVC Remote Setup

##### **Step 1: Create a GCS Bucket via the GCP Console**

1. **Navigate to Cloud Storage:** Open the left-hand navigation menu (hamburger icon) and go to **Cloud Storage > Buckets**.
2. **Create the Bucket:**
    - Click the **+ Create** button.
    - **Name:** Enter a globally unique name (e.g., `trip-duration-dvc-storages`).
    - **Location Type:** Choose **Region** (cheaper, best for a specific geographic area) or **Multi-region** (higher availability).
    - **Storage Class:** Select **Standard** (best for frequently accessed pipeline data).
    - **Access Control:** Select **Uniform** to manage access at the bucket level.
3. Click **Create** at the bottom of the page.

##### **Step 2. Install gcloud CLI locally and Authenticate Your User Account**

1. **Windows (via Installer):** Download and run the [Google Cloud CLI Installer](https://dl.google.com/dl/cloudsdk/channels/rapid/GoogleCloudSDKInstaller.exe). Follow the setup wizard and leave the options checked to "Start Google Cloud SDK Shell" and "Run gcloud init".

2. Open your terminal or Google Cloud SDK Shell and initialize the CLI. This will open a browser window for you to log in.
```bash
gcloud init
```
- Follow the browser prompts to log in.
- Select the GCP project you created/used in Step 1.
- Select a default compute region when prompted.

3. While `gcloud init` logs *you* in, DVC needs programmatic access. You must generate Application Default Credentials (ADC) so DVC scripts can securely talk to your GCS bucket. This will open *another* browser window. Log in and click "Allow." Your local machine is now fully authenticated for DVC and DVC will automatically detect the credentials.
```bash
gcloud auth application-default login
```

##### **Step 3. Connect local DVC to GCS Bucket**

```bash
dvc remote add -d myremote gs://trip-duration-dvc-storages/dvcstore # Add the GCS Remote as default
dvc remote modify gcs-remote jobs 8 # Configure the number of parallel jobs for DVC operations to faster upload/download
dvc push
dvc pull # To push and pull your Data, artifacts, and Models to Remote Storage

git add .dvc/config
git commit -m "Add DVC remote configuration" # Commit and push the remote configuration changes
git push origin master
```

## 3. DVC Versioning with Git

1. Standard process to follow when you make changes to i. Just the **data**, Just the **code**, or **Both data and code**.
```bash
dvc status # Check if DVC-tracked data has changed or not.
dvc commit # If yes, then commit the new version of data to DVC. It finalizes the new data version and update `.dvc` files and may change `.gitignore`.
dvc push # to push the new version of data to remote storage
git status # If only data has changed, you’ll see `.dvc` files and `.gitignore`. If code also has changed, you’ll see other files too like `test/test_model.py`, `app.py`, etc.
git add . # Stage All the Changes, Commit to Git and Push to GitHub
git commit -m "Comment for updatation in data and/or code"
git push origin master
git tag -a v1.0.0 -m "Release v1.0.0 with LightGBM" # Creating a new annotated tag for the current version of code and data
git push origin v1.0.0
```
Every time you make changes and `dvc commit` then `git commit`, a new version of `data/raw.dvc` is created and committed to Git. If you have **100s of versions** of git, you'll have 100 versions of `data/raw.dvc` in Git history — but only **one physical file** in your project at a time. Meanwhile, the actual data blobs are all stored (and versioned) in local `.dvc/cache/` or your remote `GCP Cloud Storage`, **not in Git**.

2. To recreate previous versions of data and code
```bash
git log # Git Log shows commit history with DVC-tracked IDs changes. Press `q` to **exit** the log view.
git checkout <commit-id> # This moves `HEAD` to the previous commit ID without deleting later commits and updates `data/raw.dvc` to that version.
# git checkout main # use this command to return to the main branch
dvc pull # Restore the correct past Version of Data by update the `data/` folder which match the current checked-out version of `data/raw.dvc`. It fetch data from remote storage if not available in local cache. downloads data from remote storage. Equivalent to `dvc fetch` + `dvc checkout`.
dvc fetch # fetch data from remote storage and store it in local cache. It does not update the workspace. 
dvc checkout # Restore correct past Version of Data by updating the `data/` folder which match the current checked-out version of data/raw.dvc file. only updates your workspace using data already in your local cache.
```

## 4. Create and Run ML Pipeline

In experiment tracking, we track `parameters`, `hyperparameters`, `metrics`, `artifacts`(plots, environment info), `models`, `tags`(custom metadata), `environment details`, run and experiment information.

**To run the DVC pipeline, we write below files:**
- `params.yaml`: store hyperparameters and configuration variables (e.g., learning rate, number of epochs, random seed).
- `dvc.yaml`: defines the different stages of your pipeline (e.g., prepare, train, evaluate), the commands to run them, the inputs they depend on (deps), and the outputs they produce (outs, metrics, plots). It is like *shell script* but easy to write.

**DVC generated files:**
- `dvc.lock`: DVC automatically generates it when you run the pipeline using `dvc repro`. It captures the exact state of your pipeline at that moment by recording the MD5 hashes of all dependencies, parameters, and outputs.
- `*.dvc files`: This file contains the cryptographic hash of your data.
- `.dvc/config`: stores the information about your remote storage.

1. To create **DVC Pipeline** stages if not exist in `dvc.yaml` file:
```bash
dvc stage add -n preprocess -d data/raw/data.csv -o data/processed/train.csv python src/dataset.py # Define the pre-processing stage
dvc stage add -n train -d data/processed/train.csv -o models/model.pkl python src/train.py # Define your ml model training stage
dvc stage add -f -n train -d data/processed/train.csv -o models/model.pkl python src/train.py # rewrite the train stage if stage already exist in `dvc.yaml` file
dvc dag # To visualize the pipeline stages and their dependencies as a Directed Acyclic Graph (DAG)
```

2. To run multiple experiments:
```bash
dvc exp run # To run experiments (run pipeline without updating `dvc.lock` file). It automatically tracks the metrics, and logs it as a distinct, hidden, temporary Git commit reference (an "experiment," like exp-a1b2c3).
dvc exp show # To show all the experiments and their metrics
dvc exp diff <exp_id1> <exp_id2> # To compare two experiments and see the differences in metrics, parameters, and code changes
dvc metrics show # To show the metrics of all the experiments
dvc exp remove <exp_id> # To remove an experiment
dvc exp run --queue -S train_model.seed=123,108,42 -S train_model.learning_rate=0.01,0.05,0.1 # To queue up multiple experiments with different parameters and run them sequentially. It work like grid search for hyperparameter as it automatically creates multiple experiments for each combination of parameters.
dvc exp run --run-all # To run all the queued experiments sequentially.
dvc exp apply <exp_id> # It pulls the exact code, data, and parameters from your best hidden experiment into your actual workspace so you can officially git commit it.
```

3. Reproduce the best experiment of DVC pipeline to save/commit:
```bash
dvc repro # It run the dvc pipeline and overwrites `dvc.lock` file. You are expected to manually run `git add dvc.lock` and `git commit` for each `dvc repro` run.
# dvc repro -f # To force run the DVC Pipeline even if there are no changes in data or code
dvc repro -s <commit_hash> # Reproduce the pipeline at a specific commit
```

## 5. Experiments Tracking with MLflow server setup

MLflow store the experiment tracking information in `mlruns/` folder and `mlflow.db` by default if mlflow database server not setup.

> Experiment : A logical grouping of runs, often representing a specific project, model, or research question. Each experiment can have multiple runs and each experiment holds a collection of multiple runs. Example : "Fraud Detection Model Training" etc.

> Run : Running an experiment with different parameters and each run holds parameters, metrics, code version, and artifacts for one execution. Example : "Training attempt with `learning_rate=0.01` and `50 epochs`" etc.

1. Update Your Firewall (Security Group)
MLflow runs on port 5000 by default and so we need to create a new firewall rule for MLflow.

- Go to **VPC network** > **Firewall** in the GCP Console.
- Click **Create Firewall Rule**.
- **Name**: `allow-port-5000`
- **Direction of traffic**: Ingress
- **Action on match**: Allow
- **Targets**: All instances in the network
- **Source filter**: IPv4 ranges
- **Source IPv4 ranges**: `0.0.0.0/0` (This makes it publicly accessible)
- **Protocols and ports**: Select "Specified protocols and ports", check TCP, and enter `5000`.
- Click **Create**.

2. Create the New MLflow VM to run the MLflow server:
- Go to **Compute Engine** > **VM instances** and click **Create Instance**.
- **Name**: `mlflow-server-vm`
- **Region/Zone**: asia-south2
- **Machine type**: `e2-small`
- **OS and storage**: Ubuntu 22.04 LTS.
- **Security**: Under **Service Account**, select the **Compute Engine default service account**.
- **Access scopes** : Select **Allow full access to all Cloud APIs**.
- Click **Create**.

3. Install and Start MLflow on the VM
Open SSH terminal and run below commands
```bash
# Update and install basic dependencies
sudo apt update
sudo apt install -y python3-pip pipx
sudo pipx ensurepath
pipx install pipenv

# Add pipx to path for this session
export PATH="$PATH:/home/$USER/.local/bin"
echo 'export PATH=$PATH:/home/'$USER'/.local/bin' >> ~/.bashrc
source ~/.bashrc

# Set up the MLflow directory and environment
mkdir mlflow_server
cd mlflow_server
pipenv shell

# Install MLflow and the GCP Cloud Storage plugin
pipenv install setuptools
pipenv install mlflow[server]
pipenv install google-cloud-storage

# Kill the currently running server
pkill -f "mlflow server"

# Start it with the NEW IP address
nohup mlflow server -h 0.0.0.0 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root gs://trip-duration-dvc-storages/mlflow-artifacts \
  --serve-artifacts \
  --workers 1 \
  --allowed-hosts <YOUR_NEW_IP>,<YOUR_NEW_IP>:5000 \
  --cors-allowed-origins "http://<YOUR_NEW_IP>:5000" \
  > mlflow.log 2>&1 &
```

4. Install MLflow and start the server:
```bash
pip install mlflow[extras] # Install MLflow for Data Scientists, during training
mlflow ui # start MLflow web interface for Local, single-user development. Defaults to 127.0.0.1 (local only)
```

5. Connect Your Local Code to the New Server
- Add mlflow code to the python scripts you want to track and log necessary information:
```python
import mlflow # During training, logs will be saved automatically in mlruns/ folder if mlflow database server not setup.
mlflow.set_tracking_uri("http://<YOUR_NEW_VM_EXTERNAL_IP>:5000")
```
Always use mlflow.sklearn.log_model instead of mlflow.log_model as this will log the
model with additional metadata specific to sklearn models, such as the library version 
and the model's signature. This can be helpful for reproducibility and for understanding
the model's dependencies when you or others revisit it in the future.

## 6. Serving the application using [Docker](https://www.docker.com/get-started)

1. Create the necessary project files like `app.py`, `requirements.txt` etc.
2. Create and update the Docker files. Build, push and run the image.
```bash
docker init # To create Docker assets like .dockerignore, compose.yaml, README.Docker.md and Dockerfile.
# Fill the these details : `Python`, `3.13.5`, `8000` and `gunicorn 'app:app' --bind=0.0.0.0:8000`
docker build -t app:latest . # To build the Docker Image from Dockerfile
# docker compose up --build # To build the image and run the container at once
docker login # Login to your Docker Hub account
docker tag app:latest vivekkumar7171/app:latest # format the image
docker push vivekkumar7171/app:latest # push the image to the Docker Hub
docker pull vivekkumar7171/app:latest # To pull the image from the Docker Hub
docker run -d -p 8000:8000 app:latest # To run a Docker container. NOTE: Press Ctrl + C to exit the program from terminal
```
3. When the container is running, visit `http://localhost:8000` to use the app.
4. To stop the container and Docker.
```bash
wsl --shutdown # To stop Vmmem or WSL (Windows Subsystem for Linux), run the below command in PowerShell
```

### Scaling of a Multi-Container Application with Docker Compose

> NOTE: You can have multiple `Dockerfiles` in different folders, and single `compose.yml` file acts as the map to connect them all. We don't need to run `docker build -t app .` and `docker run -p 8080:8080 app` commands. The `compose.yml` file saves all those settings (ports, volumes, environment variables).

1. Create a `compose.yml` file if not already created.
```bash
docker compose up --build # only build the image
docker compose up # Look at the `compose.yml` file, Build any `Dockerfiles` that need building, Download any required images and Start all the containers in the correct order.
```
2. Update the `compose.yml` file to include a service that can be scaled.
```bash
docker compose up --scale app=3 # Docker will automatically spin up three identical copies of your `app` container to share the workload, while keeping just one `db` container.
```

## 7. CI/CD SetUp with GitHub Actions and GCP's VM

### **i.** Steps to enable CI/CD with GitHub Actions 

1. Open GitHub and visit setting
2. actions
3. general
4. Select to make CI/CT to work:
    - **Actions permissions** should be **Allow all actions and reusable workflows**
    - **Artifact and log retention** should be **90** (any depending upon need but high is best for record)
    - **Approval for running fork pull request workflows from contributors** should be **Require approval for first-time contributors**
    - **Workflow permissions** should be **Read and write permissions**
    - **select/tick the option** **Allow GitHub Actions to create and approve pull requests**
5. Save the changes

### **ii.** Create the VM Instance on GCP

1. Go to the [Google Cloud Console](https://console.cloud.google.com/).
2. Navigate to **Compute Engine** > **VM instances**.
3. Click **Create Instance**.
4. **Configure your VM:**
    -  **Name:** `github-runner-1`
    -  **Region/Zone:** Choose one close to you (e.g., `asia-south2`).
    -  **Machine type:** `e2-medium` (2 vCPU, 4GB RAM) is usually plenty for a basic runner.
    -  **Boot disk:** Ubuntu 22.04 LTS (standard for most GitHub Actions).
    -  **Identity and API access:** Under **Service Account**, select the "Compute Engine default service account" for now (or a custom one if you prefer higher security).
5. Click **Create**.

### **iii.** Get the Runner Registration Commands

1. On GitHub, go to your **Repository Settings**.
2. In the left sidebar, click **Actions** > **Runners**.
3. Click **New self-hosted runner**.
4. Select **Linux** and **x64**.
5. **Stop here:** You will see a list of commands under "Download" and "Configure." Keep this tab open; you'll copy-paste these shortly.

### **iv.** Installing the Docker Engine on the GCP's VM

In the GCP Console VM list, click the **SSH** button next to your instance. A terminal window will pop up and run the below commands.

1. To update the package list and install Docker
```bash
sudo apt-get update
sudo apt-get install -y docker.io
```

2. Grant Permissions : By default, Docker requires `sudo` privileges. Your GitHub Action runner operates as a normal user and cannot type in a password for `sudo`. You need to add your user to the "docker" group so it can run Docker commands freely.
```bash
sudo usermod -aG docker $USER
```

3. Close and open the SSH terminal again to apply the group changes.
```bash
docker ps # to confirm that the container is running successfully!
docker attach trip-duration # to see the logs of the container in real-time.
```

### **v.** Install the Runner on the VM

1. In the GCP Console VM list, click the **SSH** button next to your instance. A terminal window will pop up.
2. **Paste the "Download" commands** from the GitHub tab one by one. This downloads the runner software.
3. **Paste the "Configure" commands.**
    - When you run `./config.sh`, it will ask for a name. You can hit `Enter` for defaults.
    - It will ask for **labels**. If your workflow `.github/workflows/main.yml` uses `runs-on: self-hosted`, the default label is fine.
4. **Install as a background service** (Crucial so it doesn't stop when you close the SSH window):
```bash
sudo ./svc.sh install
sudo ./svc.sh start
```

### **vi.** Authenticate GCS Access for DVC remote storage

1. **On the VM**, verify the runner has access to GCP tools:
```bash
gcloud auth list
```
2. **Permissions:** Go to **IAM & Admin** in the GCP Console.
3. Find the Service Account used by your VM (usually `[project-number]-compute@developer.gserviceaccount.com`).
4. Click the **Edit** (pencil) icon and ensure it has the role **Storage Object Admin** so DVC can read/write to your GCS bucket.

### **vii.** Create an Artifact Registry in GCP

In the Google Cloud Console, search for **Artifact Registry** in the top search bar.

1. Click **+ Create Repository**.
2. **Name**: Give it a name (e.g., `trip-duration-docker-image-hub`).
3. **Format**: Select **Docker**.
4. **Location type**: Choose Region and select the same region where your VM is located (e.g., `asia-south2`).
5. Click **Create**.

### **viii.** Create a Service Account

Your GitHub Action (running on `ubuntu-latest`) needs permission to push images into this new registry. In GCP, we use a **Service Account** instead of an IAM User. The Service Account you created earlier `trip-duration-docker-image-hub-sa` only has permission for the Artifact Registry. It now needs permission to read your GCS bucket.

1. Go to **IAM & Admin** > **Service Accounts** in the GCP Console.
2. Click **+ Create Service Account**.
3. **Name**: Call it `trip-duration-docker-image-hub-sa` and click **Create and Continue**.
4. **Roles**: Grant it the `Artifact Registry Writer` (this allows it to push and pull images) and `Storage Object Viewer` roles (this gives it read-only access to GCS) using **Add Another Role**. Click **Done**.
5. Find your newly created service account in the list, click the three dots on the right, and select **Manage keys**.
6. Click **Add Key** > **Create new key** > Choose **JSON** > Click **Create**.
7. A JSON file will download to your computer.

### **ix.** Add Secrets to GitHub

Go to your GitHub repository Settings > Secrets and variables > Actions and add the following repository secrets:

- `GCP_CREDENTIALS`: Open the downloaded JSON file in a text editor, copy the entire contents, and paste it here.
- `GCP_PROJECT_ID`: Your GCP Project ID (found on the GCP homepage).
- `GCP_REGION`: The region you chose (e.g., `asia-south2`).
- `GCP_AR_REPO_NAME`: The name of the Artifact Registry you created (e.g., `trip-duration-docker-image-hub`).

### **x.** Update GCP's default firewall rules

1. Go to **VPC network** -> **Firewall** in the GCP Console.
2. Click **Create Firewall Rule**.
3. **Name**: `allow-port-8080`
4. **Direction of traffic**: Ingress
5. **Action on match**: Allow
6. **Targets**: All instances in the network
7. **Source filter**: IPv4 ranges
8. **Source IPv4 ranges**: `0.0.0.0/0`
9. **Protocols and ports**: Select "Specified protocols and ports", check **TCP**, and enter `8080`.
10. Click **Create**.

### **xi.** Find Your VM's External IP Address and visit the endpoint

To hit your endpoint, you need the public IP of the VM hosting the Docker container.
1. In the GCP Console, go to **Compute Engine** -> **VM instances**.
2. Locate your instance (the self-hosted runner).
3. Copy the IP address listed under the **External IP** column.
```text
http://<YOUR_VM_EXTERNAL_IP>:8080/
```

## 8. Deploy Docker Image to Kubernetes (K8s) Cluster

1. To run Kubernetes Cluster, we need `docker` and [`minikube`](https://minikube.sigs.k8s.io/docs/start/) installed.
```bash
curl -LO https://github.com/kubernetes/minikube/releases/latest/download/minikube-linux-amd64 # downloading minikube binary for linux
sudo install minikube-linux-amd64 /usr/local/bin/minikube && rm minikube-linux-amd64 # Installing the minikube
# Start docker
docker context use default` # Use the default Docker context
minikube start # To start minikube and your cluster
```

2. This starts one node with `kubectl` *helps to interact with cluster* and `minikube` *helps to deploy cluster and manage it*.
```bash
kubectl get all -A # To see all the resources in all namespaces
kubectl get pods -A # To see all pods in all namespaces
kubectl get nodes -A # To see all nodes in all namespaces
minikube add node # To add more nodes to the cluster
```

3. Download the image from Image Registry(GCP's Artifact Registry) then
```bash
minikube image load trip-duration-docker-image-hub/trip-duration:latest # Load the image to minikube cluster
kubectl apply -f kubernetes-deployment.yaml # To deploy/re-deploy the application using the kubernetes-deployment.yaml file
kubectl scale --replicas=5 deployment/app # Scales the number of replicas for the `app` Deployment to 5
kubectl get deploy # Show the deployments in the default namespace that are running including the number of replicas, available replicas, and current state.
# or
kubectl get deployments # Show the deployments in the default namespace that are running including the number of replicas, available replicas, and current state.
kubectl get pods # now you see different version of the app running
kubectl delete pod trip-duration-xxxx-xxxx # To delete a specific pod, it will be recreated by the deployment
kubectl get pods # Newly created pod will be visible with new name trip-duration-xxxx-xxxx
minikube dashboard # Open the url in browser to see Kubernetes dashboard
```
We can see one extra instance of the app running, it is called `Replica Sets` to protect from failure.
```bash
minikube service trip-duration # enable port forwarding to access the app from browser
```

```bash
kubectl logs -f trip-duration-xxxx-xxxx # To see the logs of a specific pod, it will show the logs in real-time
```

## 9. Automated Schedule to cleanup disk space Setup

```bash
# Log into your runner machine & Install the Cron Utility where the `/home/mrvivekkumar7171/` directory lives.
sudo apt-get update
sudo apt-get install cron -y

# Start and Enable the Cron Service
sudo systemctl enable cron
sudo systemctl start cron

# Save the command in file
(crontab -l 2>/dev/null; echo '0 2 * * * docker system prune -af --volumes --filter "until=24h" && find /home/mrvivekkumar7171/actions-runner/_diag/ -name "*.log" -type f -mtime +3 -delete') | crontab -

# To verify it worked or not. If you see your rule printed on the screen, you are fully set up! The cleanup will happen automatically. in the background every night

crontab -l
```

## 10. Seldon-Core on K8s cluster

1. Setup and start Kubernetes Cluster

2. Installing [`helm`](https://helm.sh/docs/intro/install/)
```bash
sudo apt-get install curl gpg apt-transport-https --yes
curl -fsSL https://packages.buildkite.com/helm-linux/helm-debian/gpgkey | gpg --dearmor | sudo tee /usr/share/keyrings/helm.gpg > /dev/null
echo "deb [signed-by=/usr/share/keyrings/helm.gpg] https://packages.buildkite.com/helm-linux/helm-debian/any/ any main" | sudo tee /etc/apt/sources.list.d/helm-stable-debian.list
sudo apt-get update
sudo apt-get install helm # installing helm
```

3. Installing [`kustomize`](https://kubectl.docs.kubernetes.io/installation/kustomize/)
```bash
docker pull registry.k8s.io/kustomize/kustomize:v5.0.0 # pull the image of kustomize
docker run registry.k8s.io/kustomize/kustomize:v5.0.0 version # run 'kustomize version'
```

4. Installing Ingress (Servicing Layer) [istio](https://istio.io/latest/docs/setup/install/helm/) and [Istio Gateway](https://artifacthub.io/packages/helm/istio-official/gateway)
```bash
helm repo add istio https://istio-release.storage.googleapis.com/charts # Add the Istio Helm Repository
helm repo update

kubectl create namespace istio-system # Create the istio-system namespace manually as Helm doesn't do it automatically
helm install istio-base istio/base -n istio-system --set defaultRevision=default --wait # Install the base CRDs or Base Chart (CRDs)
helm install istiod istio/istiod -n istio-system --wait # Install the Control Plane (istiod)
helm install istio-ingressgateway istio/gateway
```

5. Install `Seldon Core` in the `seldon-system` namespace
```bash
helm repo add seldon https://storage.googleapis.com/seldon-charts # Add the Seldon Core Helm repository
helm repo update # Update Helm repositories
kubectl create namespace seldon-system
helm install seldon-core seldon/seldon-core-operator --namespace seldon-system # Install Seldon Core Operator
```
or
```bash
helm install seldon-core seldon-core-operator --repo https://storage.googleapis.com/seldon-charts --set usageMetrics.enabled=true --set istio.enabled=true --namespace seldon-system # Install Seldon Core in the seldon-system namespace
kubectl get pods -n seldon-system # To see the running seldon-core pods
kubectl describe pods seldon-controller-manager-XXXXXXXXXXX -n seldon-system # to debug the seldon-controller-manager pod.
```

6. Running Istio
```bash
istioctl version # To check if istio is installed correctly
istioctl install --set profile=demo -y # setup istio with demo profile
kubectl create namespace seldon
# kubectl label namespace seldon istio-injection=enabled #DO NOT RUN THIS COMMAND, IT IS CAUSING DEPLOYMENT ISSUES
kubectl config set-context --current --namespace=seldon # Set namespace to seldon
```

7. Deploying the models, podmonitor and gateway.
Once the model is trained after all the training and hyper parameter tuning, push it to Cloud Storage like GCS and write the seldon deployment yaml file like `seldon-iris.yaml` and `seldon_trip_duration.yaml` with the model URI. Only `seldon-iris.yaml` and `seldon_trip_duration.yaml` are changed for different project or model, while  `seldon-gateway.yaml`, and `seldon-podmonitor.yaml` will remain the same for all the projects as they are used for routing and monitoring respectively. Also, In `seldon-iris.yaml`, we only pass model and seldon handel the containerization, while in `seldon_trip_duration.yaml`, we pass image URI and seldon handle the deployment and scaling of the model only.
```bash
kubectl apply -f seldon_trip_duration.yaml -n seldon # Deploying models
kubectl apply -f seldon-iris.yaml -n seldon # A/B Testing of models
kubectl apply -f seldon-gateway.yaml -n istio-system
```

8. Deploying Prometheus and Alertmanager for monitoring
```bash
kubectl create namespace seldon-monitoring # Create a namespace for monitoring
helm upgrade --install seldon-monitoring kube-prometheus --version 8.3.2 --set fullnameOverride=seldon-monitoring --namespace seldon-monitoring --repo https://charts.bitnami.com/bitnami
kubectl get pods -n seldon-monitoring # To check if all the pods are running in the seldon-monitoring namespace
kubectl apply -f seldon-podmonitor.yaml -n seldon-monitoring # deploying Prometheus to scrape ports named metrics from pods managed by Seldon Core.
```

9. do port forwarding to access the application and monitoring tools from the browser
```bash
kubectl port-forward -n istio-system svc/istio-ingressgateway 8003:80 # To shift into background use Ctrl+Z and then running bg command
kubectl port-forward -n seldon-monitoring svc/seldon-monitoring-prometheus 9090:9090 # Expose Prometheus to the browser on 9090 port
kubectl port-forward -n seldon-monitoring svc/seldon-monitoring-alertmanager 9093:9093
```
Now, Prometheus and Alertmanager can be accessed via port "9090" and "9093" respectively.

10. To update the model deployment, update the yaml then run the below command
```bash
kubectl apply -f seldon_trip_duration.yaml
```

11. To delete the deployment
NOTE: Seldon deployment cannot be deleted like normal pods as control manager for seldon is seldon-system not the K8s's control manager.
```bash
kubectl get sdep # To see the seldon deployments
kubectl delete pod <deployment-name> # This will not work as the pod is recreated by seldon controller manager
kubectl delete sdep <deployment-name> # To delete a seldon deployment
kubectl delete deploy seldon-controller-manager -n seldon-system # To stop new deployments, delete the controller manager
```

How to use Grafana then if seldon-core-analytics is deprecated?

## 11. Kubeflow Pipelines

Make your pipeline more dynamic by parameterizing components and version controlling your pipeline code. This ensures reproducibility and flexibility. The steps of building an automated retraining pipeline with Kubeflow Pipelines are as follows:

1. Set Up Kubeflow Pipelines on Minikube and run the Kubeflow Pipelines dashboard
```bash
minikube start
export PIPELINE_VERSION=2.0.5 # Use SET for Windows
kubectl apply -k "github.com/kubeflow/pipelines/manifests/kustomize/cluster-scoped-resources?ref=$PIPELINE_VERSION"
kubectl wait --for condition=established --timeout=60s crd/applications.app.k8s.io
kubectl apply -k "github.com/kubeflow/pipelines/manifests/kustomize/env/platform-agnostic-pns?ref=$PIPELINE_VERSION"
```

2. Write a Kubeflow Pipeline script that defines a pipeline for retraining a machine learning model.

3. Run the script using a Python environment with Kubeflow Pipelines and the required dependencies installed. It will generate a yaml file for the pipeline.

4. Navigate to the Kubeflow Pipelines dashboard and upload your pipeline's yaml file. It will create graph as pipeline in the dashboard.

5. To run the experiment, first create it and run the pipeline, providing the necessary input parameters.